# Assignment 2 — DataQualityAgent

Этот ноутбук показывает полный цикл quality-агента:

- `detect_issues()`
- `choose_strategy()`
- `fix()`
- `compare()`
- `run()` как observe → decide → act → evaluate

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "agents").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from agents.data_quality_agent import DataQualityAgent

candidate_paths = [
    ROOT / "data/interim/rewrite.parquet",
    ROOT / "data/raw/merged_raw.csv",
]
for path in candidate_paths:
    if path.exists():
        df = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
        break
else:
    raise FileNotFoundError("No input dataset found in data/interim/rewrite.parquet or data/raw/merged_raw.csv")

agent = DataQualityAgent(task_type="text_classification")
report = agent.detect_issues(df)
strategy = agent.choose_strategy(report, df)

df.shape, strategy

In [ ]:
missing_df = pd.DataFrame(
    sorted((report["missing"]["per_column"] or {}).items(), key=lambda x: -x[1]),
    columns=["column", "missing_count"],
)
missing_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

if not missing_df.empty:
    missing_df.plot(kind="bar", x="column", y="missing_count", ax=axes[0], color="goldenrod")
else:
    axes[0].text(0.5, 0.5, "No missing values", ha="center", va="center")
axes[0].set_title("Missing values")
axes[0].set_ylabel("count")

imbalance = report.get("imbalance") or {}
counts = pd.Series(imbalance.get("counts") or {})
if not counts.empty:
    counts.sort_values(ascending=False).plot(kind="bar", ax=axes[1], color="coral")
else:
    axes[1].text(0.5, 0.5, "No label distribution", ha="center", va="center")
axes[1].set_title("Class imbalance")
axes[1].set_ylabel("count")

text_col = agent.text_column
if text_col in df.columns:
    df[text_col].astype("string").fillna("").str.len().plot(kind="hist", bins=40, ax=axes[2], color="steelblue")
else:
    axes[2].text(0.5, 0.5, "No text column", ha="center", va="center")
axes[2].set_title("Text length in chars")
axes[2].set_xlabel("chars")

plt.tight_layout()
plt.show()

In [ ]:
manual_strategy = {
    "missing": "fill",
    "duplicates": "drop",
    "outliers": "keep",
}

df_manual = agent.fix(df, manual_strategy)
df_agent = agent.fix(df, strategy)
comparison_manual = agent.compare(df, df_manual)
comparison_agent = agent.compare(df, df_agent)

print("Agent reasoning:")
for line in strategy["reasoning"]:
    print("-", line)

comparison_agent

In [ ]:
result = agent.run(df)
result["comparison"]

## Обоснование лучшей стратегии

Здесь удобно написать короткое объяснение для защиты:

- почему пустой `text` и пустой `label` считаются критичными;
- почему дубликаты по `text` искажают обучение;
- почему для текстовой задачи выбросы лучше искать по длине текста, а не только по числовым колонкам;
- почему выбранная агентом стратегия лучше ручной альтернативы.